# CSE445 Machine Learning — Final Report
## Hospital-Onset Infection Prediction
### Part 2: NLP & LLM Models (X = 4 for 4 Group Members)

| # | Model | Type | Checkpoint |
|---|---|---|---|
| 1 | BioBERT | NLP | `dmis-lab/biobert-v1.1` |
| 2 | ClinicalBERT | NLP | `emilyalsentzer/Bio_ClinicalBERT` |
| 3 | BioBART | LLM | `GanjinZero/biobart-v2-base` |
| 4 | GPT-2 Fine-tuned | LLM | `openai-community/gpt2` |

**Steps covered:**
- Step 4: Train all 4 NLP/LLM models
- Step 5: Accuracy, Precision, Recall, F1 for all models
- Step 6: Model size, # parameters, training time per epoch
- Step 7: Training/Validation Loss & Accuracy curves + Confusion Matrix
- Step 8: Text data augmentation

---
> ⚠️ **IMPORTANT: Enable GPU before running!**
> - Runtime → Change runtime type → Hardware accelerator → **GPU (T4)**
> - Upload `HO_infxn_analysis.csv` to Colab


## Step 0 — Install Libraries & Check GPU

In [ ]:
!pip install transformers torch datasets accelerate sentencepiece --quiet

In [ ]:
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    GPT2Tokenizer, GPT2ForSequenceClassification,
    get_linear_schedule_with_warmup
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Using device: {DEVICE}")

## Step 1 — Load Data & Convert to Clinical Text Narratives

In [ ]:
# ── CHANGE PATH IF NEEDED ─────────────────────────────────────────────────
DATA_PATH = "HO_infxn_analysis.csv"
# If on Google Drive: DATA_PATH = "/content/drive/MyDrive/HO_infxn_analysis.csv"

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded: {df_raw.shape}")
print(df_raw['group_binary'].value_counts())

In [ ]:
def row_to_clinical_text(row, col_names):
    """Convert one patient record into a clinical narrative sentence."""
    sex_str = "male" if str(row.get("sex","")).lower() in ["m","male","1"] else "female"
    age_str = str(int(float(str(row.get("age","0")).replace(">",""))))

    abx_cols = [c for c in col_names if "_0_60" in c]
    abx_list = [c.replace("_0_60","").replace("_"," ") for c in abx_cols if row.get(c,0)==1]

    cp_cols  = ["DS_Entero_cp","ESBL_cp","CDiff_cp","VSE_cp","VRE_cp",
                "MSSA_cp","MRSA_cp","DS_PsA_cp","DR_PsA_cp"]
    pathogens = [c.replace("_cp","").replace("_"," ") for c in cp_cols if row.get(c,0)>0]

    elix_cols = [c for c in col_names if c.startswith("elix_") and c!="elix_index_mortality"]
    comorbids = [c.replace("elix_","").replace("_"," ") for c in elix_cols if row.get(c,0)==1]

    surgery = "underwent surgery" if row.get("any_surgery",0)==1 else "no surgery"
    elix_sc = str(int(row.get("elix_index_mortality",0)))
    duration = f"{float(row.get('duration',0)):.1f}"

    text = (
        f"Patient is a {age_str}-year-old {sex_str} admitted for {duration} days. "
        f"Elixhauser mortality index: {elix_sc}. "
        f"The patient {surgery}. "
        f"Antibiotics (last 60 days): {', '.join(abx_list[:5]) if abx_list else 'none'}. "
        f"Colonization pressures: {', '.join(pathogens[:3]) if pathogens else 'none'}. "
        f"Comorbidities: {', '.join(comorbids[:5]) if comorbids else 'none'}."
    )
    return text

# Use a 20k stratified sample (faster training, still representative)
df_sample = df_raw.sample(n=min(20000, len(df_raw)), random_state=42).reset_index(drop=True)
print("Generating clinical text narratives...")
df_sample["text"]  = df_sample.apply(lambda r: row_to_clinical_text(r, df_raw.columns.tolist()), axis=1)
df_sample["label"] = df_sample["group_binary"]
print(f"Done. Example:\n{df_sample['text'].iloc[0]}")

In [ ]:
# Train / Val / Test split
texts  = df_sample["text"].tolist()
labels = df_sample["label"].tolist()

X_tr_txt, X_tmp, y_tr, y_tmp = train_test_split(texts, labels, test_size=0.30, stratify=labels, random_state=42)
X_val_txt, X_te_txt, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)
print(f"Train={len(X_tr_txt)}  Val={len(X_val_txt)}  Test={len(X_te_txt)}")

## Step 2 — Text Data Augmentation (Step 8)

In [ ]:
SYNONYMS = {
    "antibiotics":    ["antimicrobials","antibacterials","anti-infective agents"],
    "surgery":        ["surgical procedure","operative intervention","operation"],
    "comorbidities":  ["co-existing conditions","concurrent diagnoses"],
    "admitted":       ["hospitalized","received inpatient care"],
    "patient":        ["individual","hospital patient","subject"],
    "infection":      ["infectious process","microbial infection"],
}

import random
random.seed(42)

def augment(text):
    for word, syns in SYNONYMS.items():
        if word in text.lower() and random.random() > 0.5:
            text = text.replace(word, random.choice(syns), 1)
    return text

# Augment training minority class (cases)
aug_texts, aug_labels = list(X_tr_txt), list(y_tr)
for txt, lbl in zip(X_tr_txt, y_tr):
    if lbl == 1:
        aug_texts.append(augment(txt))
        aug_labels.append(lbl)

print(f"Original train: {len(X_tr_txt)}  →  Augmented: {len(aug_texts)}")
print(f"Class dist: {pd.Series(aug_labels).value_counts().to_dict()}")

## Step 3 — Dataset Class & Training Infrastructure

In [ ]:
class ClinicalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids":      enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels":         torch.tensor(self.labels[idx], dtype=torch.long)}


def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable, round(total*4/1024**2, 1)   # size in MB (float32)


def train_and_eval(model_name, model, tokenizer,
                   tr_texts, tr_labels, val_texts, val_labels,
                   epochs=3, batch_size=16, lr=2e-5, max_len=128):

    total_p, train_p, size_mb = count_params(model)
    print(f"\n{'='*60}")
    print(f"Model       : {model_name}")
    print(f"Parameters  : {total_p:,}  (trainable: {train_p:,})")
    print(f"Size        : {size_mb} MB")
    print(f"{'='*60}")

    tr_ds  = ClinicalDataset(tr_texts,  tr_labels,  tokenizer, max_len)
    val_ds = ClinicalDataset(val_texts, val_labels, tokenizer, max_len)
    te_ds  = ClinicalDataset(X_te_txt,  y_te,       tokenizer, max_len)

    tr_dl  = DataLoader(tr_ds,  batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    te_dl  = DataLoader(te_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(tr_dl) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1*total_steps),
        num_training_steps=total_steps)

    history = {"train_loss":[],"val_loss":[],"train_acc":[],"val_acc":[]}
    epoch_times = []

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        tr_loss, tr_ok, tr_n = 0.0, 0, 0
        t0 = time.time()
        for batch in tr_dl:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbl  = batch["labels"].to(DEVICE)
            optimizer.zero_grad()
            out  = model(input_ids=ids, attention_mask=mask, labels=lbl)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            tr_loss += out.loss.item()*ids.size(0)
            tr_ok   += (out.logits.argmax(-1)==lbl).sum().item()
            tr_n    += ids.size(0)
        epoch_time = time.time()-t0
        epoch_times.append(epoch_time)

        # ── Validate ──
        model.eval()
        vl_loss, vl_ok, vl_n = 0.0, 0, 0
        with torch.no_grad():
            for batch in val_dl:
                ids  = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                lbl  = batch["labels"].to(DEVICE)
                out  = model(input_ids=ids, attention_mask=mask, labels=lbl)
                vl_loss += out.loss.item()*ids.size(0)
                vl_ok   += (out.logits.argmax(-1)==lbl).sum().item()
                vl_n    += ids.size(0)

        tl = tr_loss/tr_n; ta = tr_ok/tr_n
        vl = vl_loss/vl_n; va = vl_ok/vl_n
        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_acc"].append(ta);  history["val_acc"].append(va)
        print(f"  Epoch {epoch+1}/{epochs} | TrainLoss={tl:.4f} ValLoss={vl:.4f} "
              f"TrainAcc={ta:.4f} ValAcc={va:.4f} | {epoch_time:.0f}s")

    # ── Test ──
    model.eval()
    all_preds, all_probs = [], []
    with torch.no_grad():
        for batch in te_dl:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            out  = model(input_ids=ids, attention_mask=mask)
            probs = torch.softmax(out.logits, dim=-1)[:,1]
            all_preds.extend(out.logits.argmax(-1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    metrics = {
        "Model"             : model_name,
        "Accuracy"          : round(accuracy_score(y_te, all_preds), 4),
        "Precision"         : round(precision_score(y_te, all_preds, average="weighted", zero_division=0), 4),
        "Recall"            : round(recall_score(y_te, all_preds, average="weighted", zero_division=0), 4),
        "F1"                : round(f1_score(y_te, all_preds, average="weighted", zero_division=0), 4),
        "ROC_AUC"           : round(roc_auc_score(y_te, all_probs), 4),
        "Total_Params"      : total_p,
        "Trainable_Params"  : train_p,
        "Model_Size_MB"     : size_mb,
        "Avg_Time_Per_Epoch": round(np.mean(epoch_times), 1),
    }
    print(f"\n  TEST → Acc={metrics['Accuracy']} F1={metrics['F1']} AUC={metrics['ROC_AUC']}")
    return model, history, metrics, all_preds, all_probs


def plot_curves(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"Training Curves — {model_name}", fontsize=13, fontweight="bold")
    eps = range(1, len(history["train_loss"])+1)
    axes[0].plot(eps, history["train_loss"], "o-", color="#2196F3", label="Train Loss")
    axes[0].plot(eps, history["val_loss"],   "s-", color="#F44336", label="Val Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(eps, history["train_acc"], "o-", color="#4CAF50", label="Train Acc")
    axes[1].plot(eps, history["val_acc"],   "s-", color="#FF9800", label="Val Acc")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    fname = f"curves_{model_name.replace(' ','_').replace('/','_')}.png"
    plt.savefig(fname, dpi=150); plt.show()
    print(f"Saved: {fname}")

nlp_results = []
print("Infrastructure ready. Starting model training...")

## Step 4a — NLP Model 1: BioBERT

In [ ]:
CKPT = "dmis-lab/biobert-v1.1"
tok1 = AutoTokenizer.from_pretrained(CKPT)
mdl1 = AutoModelForSequenceClassification.from_pretrained(CKPT, num_labels=2).to(DEVICE)

m1, h1, r1, p1, prob1 = train_and_eval(
    "BioBERT", mdl1, tok1,
    X_tr_txt, y_tr, X_val_txt, y_val,
    epochs=3, batch_size=16, lr=2e-5
)
nlp_results.append(r1)
plot_curves(h1, "BioBERT")

# Confusion matrix
cm1 = confusion_matrix(y_te, p1, normalize="true")
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(cm1, display_labels=["Control","Case"]).plot(ax=ax, cmap="Blues", values_format=".2f")
ax.set_title("BioBERT — Normalized CM", fontweight="bold"); plt.tight_layout(); plt.show()

del mdl1; torch.cuda.empty_cache()

## Step 4b — NLP Model 2: ClinicalBERT

In [ ]:
CKPT = "emilyalsentzer/Bio_ClinicalBERT"
tok2 = AutoTokenizer.from_pretrained(CKPT)
mdl2 = AutoModelForSequenceClassification.from_pretrained(CKPT, num_labels=2).to(DEVICE)

m2, h2, r2, p2, prob2 = train_and_eval(
    "ClinicalBERT", mdl2, tok2,
    X_tr_txt, y_tr, X_val_txt, y_val,
    epochs=3, batch_size=16, lr=2e-5
)
nlp_results.append(r2)
plot_curves(h2, "ClinicalBERT")

cm2 = confusion_matrix(y_te, p2, normalize="true")
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(cm2, display_labels=["Control","Case"]).plot(ax=ax, cmap="Blues", values_format=".2f")
ax.set_title("ClinicalBERT — Normalized CM", fontweight="bold"); plt.tight_layout(); plt.show()

del mdl2; torch.cuda.empty_cache()

## Step 4c — LLM Model 1: BioBART

In [ ]:
CKPT = "GanjinZero/biobart-v2-base"
tok3 = AutoTokenizer.from_pretrained(CKPT)
mdl3 = AutoModelForSequenceClassification.from_pretrained(CKPT, num_labels=2).to(DEVICE)

m3, h3, r3, p3, prob3 = train_and_eval(
    "BioBART", mdl3, tok3,
    X_tr_txt, y_tr, X_val_txt, y_val,
    epochs=3, batch_size=8, lr=1e-5   # smaller batch for larger model
)
nlp_results.append(r3)
plot_curves(h3, "BioBART")

cm3 = confusion_matrix(y_te, p3, normalize="true")
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(cm3, display_labels=["Control","Case"]).plot(ax=ax, cmap="Blues", values_format=".2f")
ax.set_title("BioBART — Normalized CM", fontweight="bold"); plt.tight_layout(); plt.show()

del mdl3; torch.cuda.empty_cache()

## Step 4d — LLM Model 2: GPT-2 Fine-tuned

In [ ]:
CKPT = "openai-community/gpt2"
tok4 = GPT2Tokenizer.from_pretrained(CKPT)
tok4.pad_token = tok4.eos_token   # GPT-2 has no pad token by default
mdl4 = GPT2ForSequenceClassification.from_pretrained(CKPT, num_labels=2).to(DEVICE)
mdl4.config.pad_token_id = tok4.pad_token_id

m4, h4, r4, p4, prob4 = train_and_eval(
    "GPT-2 Fine-tuned", mdl4, tok4,
    X_tr_txt, y_tr, X_val_txt, y_val,
    epochs=3, batch_size=8, lr=1e-5
)
nlp_results.append(r4)
plot_curves(h4, "GPT-2 Fine-tuned")

cm4 = confusion_matrix(y_te, p4, normalize="true")
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(cm4, display_labels=["Control","Case"]).plot(ax=ax, cmap="Blues", values_format=".2f")
ax.set_title("GPT-2 Fine-tuned — Normalized CM", fontweight="bold"); plt.tight_layout(); plt.show()

del mdl4; torch.cuda.empty_cache()

## Step 5 — All NLP/LLM Results (Step 5 & 6)

In [ ]:
df_nlp = pd.DataFrame(nlp_results)
print("="*70)
print("NLP & LLM RESULTS (Steps 5 & 6)")
print("="*70)
print("\nPerformance Metrics:")
print(df_nlp[["Model","Accuracy","Precision","Recall","F1","ROC_AUC"]].to_string(index=False))
print("\nModel Size & Training Time (Step 6):")
print(df_nlp[["Model","Total_Params","Trainable_Params","Model_Size_MB","Avg_Time_Per_Epoch"]].to_string(index=False))
df_nlp.to_csv("nlp_llm_results.csv", index=False)

# Comparison bar chart
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("NLP & LLM Model Comparison — Hospital-Onset Infection",
             fontsize=14, fontweight="bold")
colors = ["#2196F3","#4CAF50","#FF9800","#9C27B0"]
for ax, metric in zip(axes.flat, ["Accuracy","Precision","Recall","F1"]):
    bars = ax.bar(df_nlp.Model, df_nlp[metric], color=colors, edgecolor="white", width=0.6)
    for bar, val in zip(bars, df_nlp[metric]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                f"{val:.4f}", ha="center", va="bottom", fontsize=10)
    ax.set_ylim(0.75, 1.0); ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.set_xticklabels(df_nlp.Model, rotation=12, ha="right")
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("nlp_llm_comparison.png", dpi=150)
plt.show()
print("Saved: nlp_llm_results.csv, nlp_llm_comparison.png")

## Step 6 — Augmented Re-run on Best NLP Model

In [ ]:
# Re-train best model (ClinicalBERT) on augmented training data
best_nlp = df_nlp.loc[df_nlp.F1.idxmax(), "Model"]
print(f"Best NLP model: {best_nlp} — re-training with augmented data...")

CKPT = "emilyalsentzer/Bio_ClinicalBERT"
tok_aug = AutoTokenizer.from_pretrained(CKPT)
mdl_aug = AutoModelForSequenceClassification.from_pretrained(CKPT, num_labels=2).to(DEVICE)

_, h_aug, r_aug, _, _ = train_and_eval(
    "ClinicalBERT [Augmented]", mdl_aug, tok_aug,
    aug_texts, aug_labels, X_val_txt, y_val,
    epochs=3, batch_size=16, lr=2e-5
)
plot_curves(h_aug, "ClinicalBERT Augmented")
print(f"\nAugmented F1 = {r_aug['F1']}  vs original = {df_nlp.loc[df_nlp.Model=='ClinicalBERT','F1'].values[0]}")
del mdl_aug; torch.cuda.empty_cache()

## ✅ Notebook 2 Complete!

**Files saved:**
- `nlp_llm_results.csv` — performance table for all 4 models
- `curves_BioBERT.png` — training curves
- `curves_ClinicalBERT.png` — training curves
- `curves_BioBART.png` — training curves
- `curves_GPT-2_Fine-tuned.png` — training curves
- `nlp_llm_comparison.png` — 4-metric bar chart

**Download all output files:** Files → right-click → Download
